# DNS Testing Notebook

## Testing

### Servers Online Test

In [33]:
import dns.resolver
import sys

def verify_a_record(domain, expected_ip, dns_server=None):
    """
    Query A record for a domain and verify it matches expected IP.
    Fails if multiple A records are returned.
    
    Args:
        domain (str): Domain name to query
        expected_ip (str): Expected IP address
        dns_server (str, optional): DNS server to query (defaults to system DNS)
    
    Returns:
        bool: True if verification passes, False otherwise
    """
    try:
        # Configure resolver
        resolver = dns.resolver.Resolver()
        if dns_server:
            resolver.nameservers = [dns_server]
        
        # Query A record
        answers = resolver.resolve(domain, 'A')
        
        # Check if we have exactly one answer
        if len(answers) != 1:
            print(f"❌ FAIL: Expected 1 A record, got {len(answers)} records")
            for i, answer in enumerate(answers):
                print(f"   Record {i+1}: {answer}")
            return False
        
        # Get the single IP address
        actual_ip = str(answers[0])
        
        # Verify it matches expected IP
        if actual_ip == expected_ip:
            print(f"✅ PASS: {domain} resolves to {actual_ip}")
            return True
        else:
            print(f"❌ FAIL: {domain} resolves to {actual_ip}, expected {expected_ip}")
            return False
            
    except dns.resolver.NXDOMAIN:
        print(f"❌ FAIL: Domain {domain} does not exist")
        return False
    except dns.resolver.NoAnswer:
        print(f"❌ FAIL: No A record found for {domain}")
        return False
    except Exception as e:
        print(f"❌ ERROR: {str(e)}")
        return False

### DNSSEC Validation Test

In [34]:
import dns.message, dns.query, dns.flags

q = dns.message.make_query("example.com", "A", want_dnssec=True)
r = dns.query.udp(q, "9.9.9.9", timeout=2)
bool(r.flags & dns.flags.AD), bool(r.flags & dns.flags.CD)
if (r.flags & dns.flags.AD):
    print('AD set, DNSSEC Validation Passed')
else:
    print('No AD, test failed')

AD set, DNSSEC Validation Passed


In [35]:
# Test cases - modify these for your needs
test_cases = [
    ("ns1.devries.tv", "74.110.167.35"),  # Replace with your known good IP
    ("g.root-servers.net", "192.112.36.4"),
]

# Run tests
print("DNS A Record Verification Tests")
print("=" * 40)

all_passed = True
for domain, expected_ip in test_cases:
    result = verify_a_record(domain, expected_ip)
    all_passed &= result
    print()

print("=" * 40)
if all_passed:
    print("🎉 All tests PASSED!")
else:
    print("⚠️  Some tests FAILED!")

DNS A Record Verification Tests
✅ PASS: ns1.devries.tv resolves to 74.110.167.35

✅ PASS: g.root-servers.net resolves to 192.112.36.4

🎉 All tests PASSED!
